# Predicting Stellar Class: Optimized Pipeline

This is a fresh, clean notebook designed to build the final model pipeline without encountering Out-of-Memory (OOM) errors. 

The previous exploratory notebook (`predicting_stellar_class.ipynb`) reached its memory limits after generating multiple large datasets (SMOTE oversampling, under-sampling, engineered color indices) and running intensive Optuna hyperparameter tuning. You can review the full history of our experiments in the original exploratory notebook on GitHub.

## Summary of Previous Findings
Through extensive experimentation in the first iteration, we mapped out the absolute best path forward. Here is what we discovered:

### 1. Data Preprocessing & Scaling
*   **Categoricals:** Dropping `id`, mapping the target class to integers (GALAXY=0, QSO=1, STAR=2), binary encoding `galaxy_population`, and one-hot encoding `spectral_type` works perfectly.
*   **Numerical Scaling:** We established a tailored scaling strategy to prevent data leakage and handle diverse distributions:
    *   `alpha`, `delta` -> `MinMaxScaler`
    *   `g`, `r`, `i` (Well-behaved bands) -> `StandardScaler`
    *   `u`, `z` (Outlier-heavy bands) -> `RobustScaler`
    *   `redshift` (Highly skewed) -> `QuantileTransformer`

### 2. Handling Class Imbalance
*   We experimented with both **SMOTE (Oversampling)** and **RandomUnderSampler**. 
*   **Conclusion:** Neither helped. SMOTE generated noisy data that blurred the decision boundaries (dropping precision), and undersampling threw away ~230,000 valuable examples of galaxies, causing overall accuracy to drop. We will stick exclusively with the **Base Dataset**.

### 3. Feature Engineering
*   We analyzed feature importances and found `spectral_type_M`, `redshift`, and `galaxy_population` to be the top 3 drivers. However, reducing the dataset to *only* these three features dropped accuracy from ~96.8% to ~88.8%. The raw photometric bands are essential.
*   We explicitly engineered **Photometric Color Indices** (`u-g`, `g-r`, `r-i`, `i-z`). Surprisingly, this slightly *decreased* performance. Modern tree ensembles like XGBoost are already highly capable of finding these relationships on their own.

### 4. Error Analysis
Our `tuned_XGBoost` model reached an accuracy of **~96.8%**. The confusion matrix revealed:
*   **STAR vs. GALAXY:** This is the model's main weakness (~6.6% error rate for stars). Distant galaxies often look like point-sources (stars) photometrically.
*   **QSO vs. GALAXY:** The secondary weakness. Quasars (QSOs) live inside host galaxies, blending their light signatures.
*   **STAR vs. QSO:** Almost zero confusion, easily separated by the `redshift` feature.

## The Goal for This Notebook
In this fresh workspace, we will implement **only the optimal, high-performance pipeline**: standard preprocessing, tailored numerical scaling, and building a final optimized model without bloating the system memory.

In [ ]:
import math
import os
# Suppress OpenMP conflicting library warning that causes crashes
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import torch
import seaborn as sns
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import optuna
import gc
from xgboost import XGBClassifier
from xgboost import plot_importance
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer

print("Libraries loaded successfully.")

## 1. Data Loading

In [ ]:
def load_data():
    '''Load the data from the local directory or Kaggle dataset.'''
    try:
        print("Trying to load data from kaggle input directory...")
        train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
        test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')
        submission = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv')
        print("Data loaded successfully from kaggle input directory.")
    except FileNotFoundError:
        print("Data files not found in kaggle input directory. \nTrying to load data from local directory...")
        data_dir = os.getcwd()+ '/data/'
        train = pd.read_csv(data_dir + 'train.csv')
        test = pd.read_csv(data_dir + 'test.csv')
        submission = pd.read_csv(data_dir + 'sample_submission.csv') 
        print("Data loaded successfully from local directory.")

    return train, test, submission

train_df, test_df, submission_df = load_data()

# Check GPU availability
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}. Setting global device to 'cuda'.")
    GLOBAL_DEVICE = 'cuda'
else:
    print("No GPU detected. Setting global device to 'cpu'.")
    GLOBAL_DEVICE = 'cpu'


## 2. Optimized Preprocessing Pipeline

We apply the exact optimal preprocessing and scaling steps identified in the exploratory notebook. By keeping this pipeline clean, we minimize our memory footprint.

In [ ]:
def preprocess_data(df, is_train=True):
    temp_df = df.copy()
    if 'id' in temp_df.columns:
        temp_df = temp_df.drop(columns=['id'])
        
    if is_train and 'class' in temp_df.columns:
        class_mapping = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
        temp_df['class'] = temp_df['class'].map(class_mapping)
        
    galaxy_population_mapping = {'Red_Sequence': 0, 'Blue_Cloud': 1}
    temp_df['galaxy_population'] = temp_df['galaxy_population'].map(galaxy_population_mapping)
    temp_df = pd.get_dummies(temp_df, columns=['spectral_type'], drop_first=True, dtype=int)
    
    return temp_df

def apply_scaling(X_train, X_test=None):
    scalers = {
        'minmax': MinMaxScaler().fit(X_train[['alpha', 'delta']]),
        'standard': StandardScaler().fit(X_train[['g', 'r', 'i']]),
        'robust': RobustScaler().fit(X_train[['u', 'z']]),
        'quantile': QuantileTransformer().fit(X_train[['redshift']])
    }
    
    def transform(df):
        temp_df = df.copy()
        temp_df[['alpha', 'delta']] = scalers['minmax'].transform(temp_df[['alpha', 'delta']])
        temp_df[['g', 'r', 'i']] = scalers['standard'].transform(temp_df[['g', 'r', 'i']])
        temp_df[['u', 'z']] = scalers['robust'].transform(temp_df[['u', 'z']])
        temp_df[['redshift']] = scalers['quantile'].transform(temp_df[['redshift']])
        return temp_df
    
    X_train_scaled = transform(X_train)
    X_test_scaled = transform(X_test) if X_test is not None else None
    
    return X_train_scaled, X_test_scaled, scalers

# 1. Preprocess categorical features
train_processed = preprocess_data(train_df, is_train=True)
test_processed = preprocess_data(test_df, is_train=False)

# 2. Split into Train/Validation
X = train_processed.drop(columns=['class'])
y = train_processed['class']
X_train_raw, X_val_raw, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Align columns of test_processed to X_train_raw
test_processed = test_processed.reindex(columns=X_train_raw.columns, fill_value=0)

# 4. Apply strict scaling to prevent data leakage
X_train, X_val, scalers = apply_scaling(X_train_raw, X_val_raw)
_, test_X, _ = apply_scaling(X_train_raw, test_processed)

print("Data loaded, preprocessed, and scaled successfully!")

## 3. Hyperparameter Tuning with Optuna

We use Optuna to find the best hyperparameters for XGBoost. To ensure we don't run into Out-of-Memory (OOM) issues, we explicitly call `gc.collect()` and delete variables after each trial.

In [ ]:
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400, step=100),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'tree_method': 'hist',
        'device': GLOBAL_DEVICE,
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = XGBClassifier(**param)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    
    del model, y_pred
    gc.collect()
    
    return acc

print("\n--- Starting Optuna Tuning for XGBoost ---")
study = optuna.create_study(direction='maximize', study_name="XGBoost Tuning")
study.optimize(objective, n_trials=10, gc_after_trial=True)

print(f"\nBest Trial Accuracy: {study.best_value:.5f}")
print("Best Params:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")


## 4. Final Evaluation & Submission

With the best parameters found, we train the final model on the training set, validate it, and generate the predictions for `submission.csv`.

In [ ]:
# Train the final model
best_params = study.best_params
best_params.update({'tree_method': 'hist', 'device': GLOBAL_DEVICE, 'random_state': 42, 'n_jobs': -1})

final_model = XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

# Validate the final model
y_val_pred = final_model.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)
print(f"\nFinal Validation Accuracy: {val_acc:.5f}")

# Confusion Matrix on Validation Set
cm = confusion_matrix(y_val, y_val_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['GALAXY', 'QSO', 'STAR'])
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title(f"Confusion Matrix for Final XGBoost")
plt.show()

# Make predictions for submission
test_predictions = final_model.predict(test_X)
reverse_class_mapping = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}
submission_df['class'] = pd.Series(test_predictions).map(reverse_class_mapping)

# Save submission file
submission_df.to_csv('submission.csv', index=False)
print("Submission saved successfully to 'submission.csv'!")
submission_df.head()
